# Generating Atom-JEPA Embeddings for a Molecule

Run with a Python environment containing this project's dependencies (including PyTorch and RDKit). 
Edit `SMILES` and `CHECKPOINT` below. The checkpoint is the pretrained **Atom-JEPA context encoder**, without a fine-tuning head. `"molecules"` downloads the released encoder from [Hugging Face](https://huggingface.co/atom-jepa/atom-jepa) on first use (needs `huggingface_hub` and `safetensors`); a local checkpoint path also works.

The 3D conformers are here generated with ETKDGv3 with relaxation through MMFF94 like in the paper.

### Setup

In [1]:
from pathlib import Path
import sys

# Works when the notebook starts in the repository root or notebooks/.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "models/jepa_equiformer.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from rdkit import Chem
from rdkit.Chem import AllChem
from data.core.collate import GraphCollator
from finetuning.common import load_pretrained_encoder, move_batch

SMILES = "CC(=O)Oc1ccccc1C(=O)O"  # Aspirin
CHECKPOINT = "molecules"  # released encoder (huggingface.co/atom-jepa/atom-jepa), or a local .pt path
N_CONFORMERS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


### Generate 3D Conformers

In [2]:
mol = Chem.MolFromSmiles(SMILES)
if mol is None:
    raise ValueError(f"Invalid SMILES: {SMILES}")
mol = Chem.AddHs(mol)

params = AllChem.ETKDGv3()
params.randomSeed = 42
params.pruneRmsThresh = -1.0
conformer_ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=N_CONFORMERS, params=params))
if len(conformer_ids) != N_CONFORMERS:
    raise RuntimeError(f"Generated {len(conformer_ids)} of {N_CONFORMERS} conformers.")
if not AllChem.MMFFHasAllMoleculeParams(mol):
    raise ValueError("MMFF94 does not support all atoms in this molecule.")
optimization = AllChem.MMFFOptimizeMoleculeConfs(mol, numThreads=1, maxIters=500)
if any(status != 0 for status, energy in optimization):
    print("Some conformers did not converge; their last coordinates will be used.")

atomic_numbers = torch.tensor([atom.GetAtomicNum() for atom in mol.GetAtoms()])
samples = [
    {"atomic_numbers": atomic_numbers,
     "node_coordinates": torch.tensor(mol.GetConformer(cid).GetPositions(), dtype=torch.float32)}
    for cid in conformer_ids
]
print(f"Generated {len(samples)} conformers, each with {mol.GetNumAtoms()} atoms (including H).")


Generated 10 conformers, each with 21 atoms (including H).


### Show Conformer

In [3]:
import plotly.graph_objects as go

# Display the first generated conformer without changing its coordinates.
conformer = mol.GetConformer(conformer_ids[0])
positions = conformer.GetPositions()
symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
colors = {"H": "#dce3eb", "C": "#475569", "N": "#3b82f6", "O": "#ef4444",
          "F": "#22c55e", "Cl": "#22c55e", "Br": "#a16207", "I": "#a855f7",
          "S": "#eab308", "P": "#f97316"}

bond_x, bond_y, bond_z = [], [], []
for bond in mol.GetBonds():
    start, end = positions[bond.GetBeginAtomIdx()], positions[bond.GetEndAtomIdx()]
    for values, a, b in zip((bond_x, bond_y, bond_z), start, end):
        values.extend([float(a), float(b), None])

fig = go.Figure([
    go.Scatter3d(x=bond_x, y=bond_y, z=bond_z, mode="lines",
                 line=dict(color="#94a3b8", width=7), hoverinfo="skip"),
    go.Scatter3d(x=positions[:, 0], y=positions[:, 1], z=positions[:, 2],
                 mode="markers", text=[f"{symbol} · atom {i}" for i, symbol in enumerate(symbols)],
                 hovertemplate="%{text}<extra></extra>",
                 marker=dict(size=[6 if symbol == "H" else 11 for symbol in symbols],
                             color=[colors.get(symbol, "#a855f7") for symbol in symbols],
                             opacity=1, line=dict(color="#ffffff", width=1))),
])
fig.update_layout(
    title=dict(text="First 3D conformer · drag to rotate, scroll to zoom", x=0.5),
    height=480, showlegend=False, paper_bgcolor="#f8fafc",
    margin=dict(l=0, r=0, b=0, t=55),
    scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False),
               zaxis=dict(visible=False), aspectmode="data", bgcolor="#f8fafc",
               camera=dict(eye=dict(x=1.5, y=1.5, z=1.1))),
)
fig.show(config={"displaylogo": False})


### Setup Model

In [4]:
encoder, config, checkpoint = load_pretrained_encoder(str(CHECKPOINT), DEVICE)
del checkpoint
encoder.eval()
encoder.set_grad_checkpointing(False)
if atomic_numbers.max().item() >= config.max_num_elements:
    raise ValueError("This molecule contains an element outside the encoder's vocabulary.")
collate = GraphCollator(cutoff=config.max_radius, max_num_elements=config.max_num_elements)
print(f"Device: {DEVICE}; embedding width: {config.num_channels}; atom pooling: {config.readout_pool}")


Device: cpu; embedding width: 256; atom pooling: mean


### Compute Embeddings

In [8]:
# Small inference batches keep memory use modest; no training or compilation needed.
with torch.inference_mode():
    conformer_embeddings = torch.cat([
        encoder(move_batch(collate(samples[start:start + 2]), DEVICE)).float().cpu()
        for start in range(0, N_CONFORMERS, 2)
    ])
assert torch.isfinite(conformer_embeddings).all(), "Non-finite encoder output"

print("Per-conformer embeddings:", tuple(conformer_embeddings.shape))  # (10, embedding width)
print("First molecular embedding:", tuple(conformer_embeddings[0].shape))   # (embedding width,)

Per-conformer embeddings: (10, 256)
First molecular embedding: (256,)
